In [10]:
import pandas as pd
from sklearn.utils import resample
import os

#### 언더샘플링 예제

In [2]:
df = pd.read_parquet('./data/train/1_회원정보_train.parquet')

In [3]:
# 1. is_C 컬럼 생성
df['is_C'] = (df['Segment'] == 'C').astype(int)

In [4]:
df['is_C']

0          0
1          0
2          1
3          0
4          0
          ..
2399995    0
2399996    0
2399997    1
2399998    0
2399999    0
Name: is_C, Length: 2400000, dtype: int32

In [5]:
# 2. C와 Others 분리
df_c = df[df['is_C'] == 1].copy()
df_others = df[df['is_C'] == 0].copy()

print(f"C 샘플 수: {len(df_c)}, Others 샘플 수: {len(df_others)}")

# 3. Others에서 C만큼 랜덤 샘플링
df_others_sampled = resample(
    df_others,
    replace=False,
    n_samples=len(df_c),
    random_state=42
)

# 4. 병합 후 셔플 (선택)
df_balanced = pd.concat([df_c, df_others_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"균형 데이터셋 생성 완료: {df_balanced.shape}")

C 샘플 수: 127590, Others 샘플 수: 2272410
균형 데이터셋 생성 완료: (255180, 79)


In [9]:
df_balanced['is_C'].value_counts()

is_C
1    127590
0    127590
Name: count, dtype: int64

#### C vs Other 언더샘플링

In [12]:
# 파일 경로 리스트
paths = [
    './data/train/1_회원정보_train.parquet',
    './data/train/2_신용정보_train.parquet',
    './data/train/3_승인매출정보_train.parquet',
    './data/train/4_청구입금정보_train.parquet',
    './data/train/5_잔액정보_train.parquet',
    './data/train/6_채널정보_train.parquet',
    './data/train/7_마케팅정보_train.parquet',
    './data/train/8_성과정보_train.parquet'
]

In [16]:
# Segment 기준 정보
segment_df = pd.read_parquet('./data/train/1_회원정보_train.parquet')[['ID', '기준년월', 'Segment']]
segment_df['ID'] = segment_df['ID'].astype(str)
segment_df['기준년월'] = segment_df['기준년월'].astype(str)

# 출력 폴더
os.makedirs('./undersampled', exist_ok=True)

for path in paths:
    name = os.path.basename(path).replace('_train.parquet', '')
    df = pd.read_parquet(path)
    
    # ID, 기준년월 문자열 처리
    df['ID'] = df['ID'].astype(str)
    if '기준년월' in df.columns:
        df['기준년월'] = df['기준년월'].astype(str)
    
    # Segment 병합
    if 'Segment' not in df.columns:
        df = pd.merge(df, segment_df, on=['ID', '기준년월'], how='left')

    if df['Segment'].isna().all():
        print(f"[경고] {name} Segment 병합 실패 → 건너뜀")
        continue

    # is_C 컬럼 생성
    df['is_C'] = (df['Segment'] == 'C').astype(int)

    # C vs Others 분리
    df_c = df[df['is_C'] == 1].copy()
    df_not_c = df[df['is_C'] == 0].copy()

    if len(df_c) == 0 or len(df_not_c) == 0:
        print(f"[주의] {name} 클래스 분리 실패 → 건너뜀")
        continue

    # 언더샘플링
    df_not_c_sampled = resample(
        df_not_c,
        replace=False,
        n_samples=len(df_c),
        random_state=42
    )

    # 병합 및 저장
    df_balanced = pd.concat([df_c, df_not_c_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_balanced.to_csv(f'./undersampled/{name}_balanced.csv', index=False, encoding='utf-8-sig')
    print(f"[완료] {name} → {len(df_balanced)}건 저장 완료")

print("\n모든 파일 언더샘플링 완료 (폴더: ./undersampled)")

[완료] 1_회원정보 → 255180건 저장 완료
[완료] 2_신용정보 → 255180건 저장 완료
[완료] 3_승인매출정보 → 255180건 저장 완료
[완료] 4_청구입금정보 → 255180건 저장 완료
[완료] 5_잔액정보 → 255180건 저장 완료
[완료] 6_채널정보 → 255180건 저장 완료
[완료] 7_마케팅정보 → 255180건 저장 완료
[완료] 8_성과정보 → 255180건 저장 완료

모든 파일 언더샘플링 완료 (폴더: ./undersampled)


In [17]:
df_balanced['is_C'].value_counts()

is_C
1    127590
0    127590
Name: count, dtype: int64